# Staged RL Training for Qwen2.5-VL

This notebook copies the uploaded code dataset into `/kaggle/working`, installs dependencies, and runs one explicit training phase.

In [ ]:
import os, shutil, subprocess, glob, json, tarfile

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/kaggle/working/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

matches = []
for root, _, files in os.walk("/kaggle/input"):
    if "rl_gspo_qwen2_5vlm_test3.py" in files:
        matches.append(root)
if not matches:
    raise FileNotFoundError(f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}")

src = sorted(matches)[0]
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(src, WORKDIR)
for archive_name in ("staged_rl.tar", "tests.tar"):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)
print("Using code dataset", src)
print("Copied code to", WORKDIR)
print(sorted(os.listdir(WORKDIR)))

In [ ]:
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
print("Venv ready:", VENV_PYTHON)

In [ ]:
compat_script = """
import numpy
import scipy
import sklearn
import vllm
from vllm.sampling_params import GuidedDecodingParams
from trl import GRPOTrainer
print('numpy version:', numpy.__version__)
print('scipy version:', scipy.__version__)
print('sklearn version:', sklearn.__version__)
print('vllm version:', vllm.__version__)
print('compat imports OK')
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)

In [ ]:
PHASE = "phase_a"
HARDWARE_PROFILE = "kaggle_t4"
RESUME = None
MAX_EVAL_EXAMPLES_PER_SUBSET = 8

cmd = [
    VENV_PYTHON,
    "rl_gspo_qwen2_5vlm_test3.py",
    "--phase", PHASE,
    "--hardware-profile", HARDWARE_PROFILE,
    "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
]
if RESUME:
    cmd.extend(["--resume", RESUME])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR)

In [ ]:
outputs_root = os.path.join(WORKDIR, "outputs_staged")
collected = []
for root, _, files in os.walk(outputs_root):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected)[-50:]:
    print(path)